# 00 Read the data

This notebook opens the seven released NetCDF files and shows their content. Set the file paths once in
`config/user_paths.json` (see the README).

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from essd_cz.config import load_config

CONFIG = ROOT / 'config' / 'user_paths.json'   # copy of config/user_paths.example.json with your paths
cfg = load_config(CONFIG)
paths, OUT = cfg['datasets'], cfg['output_dir']


def run(script, *args):
    """Run a script of the repository with the configuration of this notebook."""
    cmd = [sys.executable, str(ROOT / script), '--config', str(CONFIG), *map(str, args)]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

## Check the files
`scripts/check_files.py` checks names, dimensions, entity counts and the time axis of the seven files.

In [ ]:
run('scripts/check_files.py')

## Raw and quality controlled CML signals
TX and RX are in dBm. `trsl` is the total signal loss (TX - RX) after quality control; `qc_flag` marks masked,
blackout-reconstructed (bit 8) and interpolated (bit 16) minutes.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

raw = xr.open_dataset(paths['raw'])
qc = xr.open_dataset(paths['qc'])
print('RAW:', dict(raw.sizes), '| QC:', dict(qc.sizes))
cml, day = '45035_45097_2740', slice('2021-07-13 12:00', '2021-07-14 01:00')
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for s in (0, 1):
    r = raw[['tx', 'rx']].sel(cml_id=cml, sublink_id=s, time=day)
    (r['tx'] - r['rx']).plot(ax=ax[0], label=f'raw TX - RX, sublink {s}')
    qc['trsl'].sel(cml_id=cml, sublink_id=s, time=day).plot(ax=ax[1], label=f'QC trsl, sublink {s}')
for a in ax:
    a.legend(); a.set_title('')
ax[0].set_ylabel('dB'); ax[1].set_ylabel('dB')
plt.tight_layout()

## Rainfall
The released CML rainfall is at 1 minute (mm/h). The 5-minute series of the paper are derived from it with
`scripts/derive_5min.py`.

In [ ]:
rain = xr.open_dataset(paths['rain'])
print(dict(rain.sizes))
rain['R_sublink_1min'].sel(cml_id=cml, time_1min=day).plot.line(x='time_1min', figsize=(10, 3))
plt.ylabel('mm/h');

## Reference data
Rain gauges (`rr`, mm per 10 min), radar along the CML paths (`rr`, mm per 5 min), temperature (`T`) and relative
humidity (`H`) at 10 min.

In [ ]:
for role in ('gauge', 'radar', 'temperature', 'humidity'):
    with xr.open_dataset(paths[role]) as ds:
        print(f'{role:12s}', dict(ds.sizes), list(ds.data_vars))

## Map of the CML paths

In [ ]:
import numpy as np
from matplotlib.collections import LineCollection
fig, ax = plt.subplots(figsize=(9, 5))
segs = np.stack([np.c_[raw.site_a_longitude, raw.site_a_latitude], np.c_[raw.site_b_longitude, raw.site_b_latitude]], axis=1)
ax.add_collection(LineCollection(segs, colors='k', linewidths=0.5))
g = xr.open_dataset(paths['gauge'])
ax.plot(g['lon'], g['lat'], '^', color='tab:orange', ms=4, label='rain gauge')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude'); ax.legend()
ax.set_title(f"{raw.sizes['cml_id']} CMLs of the raw dataset");